# Language Classifier (Arabic vs English)
Trains a simple classical model that tells Arabic and English text apart.
Put both datasets in a `data/` folder next to this notebook before running.

In [ ]:
import pandas as pd
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from preprocessing_pipeline import clean_for_language_detection


## Load both datasets
We don't care about sentiment here, just the raw text and which language it's in.

In [ ]:
ar_df = pd.read_csv("data/arabic_customer_reviews.csv")
en_df = pd.read_csv("data/imdb_reviews.csv")

# datasets on Kaggle don't always use the same column name, so just grab
# whichever text column exists
ar_text_col = "review" if "review" in ar_df.columns else ar_df.columns[0]
en_text_col = "review" if "review" in en_df.columns else en_df.columns[0]

ar_texts = ar_df[ar_text_col].astype(str)
en_texts = en_df[en_text_col].astype(str)

texts = pd.concat([ar_texts, en_texts], ignore_index=True)
labels = ["ar"] * len(ar_texts) + ["en"] * len(en_texts)

print(len(texts), "total rows")


## Clean the text and split

In [ ]:
cleaned = texts.apply(clean_for_language_detection)

X_train, X_test, y_train, y_test = train_test_split(
    cleaned, labels, test_size=0.2, random_state=42, stratify=labels
)


## Vectorize + train
Character n-grams work well for language detection since they pick up on spelling/alphabet patterns rather than needing full words.

In [ ]:
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=20000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = MultinomialNB()
model.fit(X_train_vec, y_train)


In [ ]:
preds = model.predict(X_test_vec)
print("accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))


## Save weights

In [ ]:
with open("Language_classifier_weights.pkl", "wb") as f:
    pickle.dump({"vectorizer": vectorizer, "model": model}, f)

print("saved Language_classifier_weights.pkl")
